# Example: Transcriptional Dynamics

The following applies minibatch and adaptive minibatch SMC ABC to the problem of estimating parameters from a transcriptional dynamics snapshot model of smFISH (single-molecule Fluorescent in situ Hybridization). The heterogeneity in these models is given by the shape (length L in this case) and transcription start site (z). The model is of bursty expression, following:


Will be filled out later

In [ ]:
## Import smc abc class and schemes
from smc_abc import smc_abc_iterator as abc_iter
import smc_abc_schemes as schemes
import smc_abc_utils as utils
import numpy as np

from matplotlib import pyplot as plt

#A class that generates smfish transcriptional dynamics snapshots
import snapshots
smfish_sims = snapshots.snapshot_simulator()

#import seaborn as sns
#import pandas as pd


cwd appended to system path


In [ ]:
# Model Parameters
sample_size = 300
kplus = 30
kminus = 10
rburst = 10
diffusivity = 0.1

true_params = np.array([kplus,rburst,diffusivity])
num_params = 3

T,dt = 5,0.01

# The heterogeneities below allow for enough non-zero count RNA. 
# parameters are chosen so mean length is 1 and mean site is 0.5 of the length
bp,gp = 2,4
lengths = np.random.gamma(shape = gp, scale = 1/gp, size = (sample_size,))
sites = np.random.beta(a = bp,b = bp, size = (sample_size,)) * lengths

length_index = np.arange(sample_size)
print(f"Length Mean: {np.mean(lengths)}. Variance: {np.var(lengths)}")
print(f"Site Mean : {np.mean(sites)}. Variance: {np.var(sites)}")


# Data generation
def bndry_func(x,ind,i=length_index):
    return (x > 0) & (x < lengths[i[ind]])
data_ = smfish_sims.simulate(kplus, kminus, rburst, diffusivity, T, dt, sites, bndry_func)
data = np.array([np.asarray(d) for d in data_],dtype=object)

# Model setup: Requires parameter vector for the particles (p) and index vector for the minibatch indices (i) both 1d
def model(p,Bi):
    z = sites[Bi,]
    bfunc = lambda x,ind: bndry_func(x,ind,i=Bi)
    obs_data = data[Bi] #alternatively, [data[j] for j in Bi]
    sim = np.array(smfish_sims.simulate(p[0], kminus, p[1], p[2], T, dt, z, bfunc),dtype = object)
    return sim, obs_data


#### Estimation parameters
alpha = 0.5
num_particles = 500
cores = 4

#### Prior
# The prior can be built in two ways. The utility function to create the uniform prior takes a matrix shaped (n,2), where each row are the bounds.
# Alternatively, you can construct one. It's a tuple containing
# (prior sampler, prior domains (shaped like utils input), vectorized density function)
prior = utils.uniform_prior([[1,100],[1,100],[0,1.0]])


#### Statistics function
# Statistics must be per-observation, being shaped like (N_batch_size,N_stats)
# Here's an example that calculates counts, squared deviation from the mean count, and root mean squared deviation
def stats_func_td(x):
    counts = np.array([u.shape[0] for u in x], dtype=float)
    mean_counts = np.mean(counts)
    var_influence = (counts - mean_counts)**2
    rmsd = np.array([np.std(u) if u.size >=2 else 0 for u in x], dtype=float)
    stats = np.vstack((counts, var_influence, rmsd)).T
    return stats
def stats_func(x,y,sf = stats_func_td):
    return sf(x),sf(y)


#### Distance function
# The distance function takes in the simulations, their corresponding obervations, and a threshold and outputs distance and (distance < threshold)
# 'mahalanobis' is the default.


#### Stopping criteria
# A wrapper in the smc_abc class exists that takes in a scheme loop and stopping function

## Stopping criterion
threshold = 0.35
time_threshold = np.inf
accept_threshold = 0
def threshold_stop(est,threshold = 1,time_threshold=np.inf,accept_threshold=0.0):
    criteria = stop(est,"current_alpha_threshold",threshold) \
        and stop(est,"total_time",time_threshold,compare=lambda x,y: x >= y) \
        and stop(est,"acceptance_rate",accept_threshold)
    return criteria
stop_func = lambda est: threshold_stop(est, threshold = threshold,time_threshold=time_threshold,accept_threshold=accept_threshold)


#### input parameters: IMPORTANT: it's convenient to pass this to the scheme initializer instead of typing out all of the parameters.
# This isn't necessary, but it saves time.
init_params = {
    'data':data,
    'model':model,
    'stats_func':stats_func,
    'prior':prior,
    'num_particles':num_particles,
    'cores':cores,
    'alpha':alpha,
    'batch_size_min':2,
    'seed':0
}

0.2752105239321874 0.11423878194784667
0
1


## Constant Minibatch SMC ABC